# 13 · Serving check: real model behind the gateway, threshold tuning (Step 5.1, D-060)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

1. **Offline predictions on val (~12 min):** E5 on the val set with the evaluation script, the reference for step 5.
2. **Start the real model server** (vLLM, OpenAI-style API, E5 adapter) and the **gateway** in the background (~5 min).
3. **Gateway check:** health, adapters, and a few requests.
4. **Replay** every val and test question through the served path: the photo is prepared exactly as the gateway does, and all three stages run on the server (~35 min).
5. **Parity and tuning (~1 min, CPU):** do served replies match the offline ones? Which decline threshold works best on val? Then that threshold is applied unchanged to test.

No training, no W&B key. If Colab disconnects, **Run all** again: the offline step is skipped once done, and the replay continues where it stopped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
MODEL = "Qwen/Qwen3-VL-4B-Instruct"
ADAPTER = f"{DRIVE_ROOT}/checkpoints/e5-multi-lora/adapter"
OFFLINE_VAL = f"{RUNS}/e5-multi-lora-val"
OUT = f"{RUNS}/serve-check"
MODEL_URL, GATEWAY_URL = "http://127.0.0.1:8000", "http://127.0.0.1:8090"  # Colab itself uses 8080


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


def stop_servers():
    """Stop model servers and gateways left by an earlier run in this session; wait for the GPU."""
    sh('pkill -f "[v]llm serve"; pkill -f "[v]izwiz.serve.gateway"; sleep 2', check=False)
    used = 0
    for _ in range(60):
        used = int(
            subprocess.run(
                "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                shell=True,
                capture_output=True,
                text=True,
            ).stdout.split()[0]
        )
        if used < 2000:
            break
        time.sleep(2)
    print(f"GPU memory in use: {used} MiB")


def get(url, timeout=5):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.status, json.loads(r.read())


os.environ["HF_HOME"] = "/content/hf_cache"
drive.mount("/content/drive")
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "E5 adapter not found on Drive"
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
stop_servers()  # safe Run all: nothing from an earlier run keeps the GPU or the ports

**Paste back:** the commit line and the GPU line (must say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e '{REPO_DIR}[serve]' -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")
sh(f"{PY} -c 'import fastapi, prometheus_client, multipart; print(\"gateway libraries OK\")'")

## 3 · Unzip photos (~6 min)
Val photos come from the official train zip, test photos from the official val zip.

In [ ]:
%%time
import hashlib

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_val.jsonl", "vqa_test.jsonl", "quality_val.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train val")

## 4 · Offline predictions on val (~12 min)
The evaluation script with E5 on the val set: the reference the served replies are compared with. Skipped when already done. The model download (~9 GB) happens here.

In [ ]:
%%time
done = (
    set(json.load(open(f"{OFFLINE_VAL}/metrics.json"))["tasks"])
    if os.path.exists(f"{OFFLINE_VAL}/metrics.json")
    else set()
)
todo = [t for t in ("answer", "answerability", "quality") if t not in done]
if not todo:
    print("offline val predictions already done: skipped")
else:
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split val --tasks {' '.join(todo)} --adapter {ADAPTER} --run-id e5-multi-lora-val --out {OFFLINE_VAL}"
    )

**Paste back:** the three result lines (`[answer] vqa_accuracy`, `[answerability] average_precision`, `[quality] macro_f1`).

## 5 · Start the model server and the gateway (~5 min)
Safe to rerun: servers left from an earlier run are stopped first. Both run in the background; their logs go to `/content/vllm.log` and `/content/gateway.log`. `--generation-config vllm` stops the model's own default sampling settings from applying, so decoding matches the offline evaluation.

In [ ]:
%%time
stop_servers()  # a rerun of this cell would otherwise find the old servers on the GPU and ports
env = {
    **os.environ,
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
    "VLLM_USE_FLASHINFER_SAMPLER": "0",
}  # R-002, D-020
vllm_cmd = (
    f"{VENV}/bin/vllm serve {MODEL} --dtype bfloat16 --max-model-len 4096 --gpu-memory-utilization 0.85 "
    f"--enable-lora --lora-modules e5-multi-lora={ADAPTER} --max-lora-rank 16 --max-loras 1 "
    f"--limit-mm-per-prompt '{{\"image\": 1}}' --generation-config vllm --seed 0 --port 8000"
)
vllm_proc = subprocess.Popen(
    vllm_cmd, shell=True, env=env, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT
)
t0 = time.time()
while True:
    try:
        status, models = get(f"{MODEL_URL}/v1/models")
        break
    except Exception:
        if vllm_proc.poll() is not None or time.time() - t0 > 900:
            sh("tail -n 40 /content/vllm.log", check=False)
            raise RuntimeError("Model server did not start. Paste the log lines above.") from None
        time.sleep(5)
print(f"model server up after {time.time() - t0:.0f} s; models:", [m["id"] for m in models["data"]])
sh(
    "grep -E 'Model loading took|Available KV cache memory|GPU KV cache size' /content/vllm.log | sed 's/^.*\\] //'",
    check=False,
)

gw_env = {**env, "VIZWIZ_MODEL_URL": MODEL_URL}
gw_proc = subprocess.Popen(
    f"{VENV}/bin/uvicorn --factory vizwiz.serve.gateway:create_app --host 127.0.0.1 --port 8090",
    shell=True,
    env=gw_env,
    cwd=REPO_DIR,
    stdout=open("/content/gateway.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        print("GATEWAY HEALTH:", get(f"{GATEWAY_URL}/health")[1])
        break
    except Exception:
        time.sleep(1)
else:
    sh("tail -n 40 /content/gateway.log", check=False)
    raise RuntimeError("Gateway did not start. Paste the log lines above.")

**Paste back:** the `model server up` line, the memory lines, and the `GATEWAY HEALTH` line.

## 6 · Gateway check with the real model (~1 min)
Synthetic photos first (safe to print the full reply), then 30 val photos where only the status and timing are printed.

In [ ]:
import io
import random
import statistics
from pathlib import Path

import requests
from PIL import Image


def synthetic(kind):
    rng = random.Random(0)
    if kind == "flat":
        img = Image.new("RGB", (640, 480), (128, 128, 128))
    else:
        img = Image.frombytes(
            "RGB", (640, 480), bytes(rng.randint(0, 255) for _ in range(640 * 480 * 3))
        )
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    return buf.getvalue()


def assist(image_bytes, question):
    r = requests.post(
        f"{GATEWAY_URL}/v1/assist",
        files={"image": ("p.jpg", image_bytes, "image/jpeg")},
        data={"question": question},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()


print("ADAPTERS:", get(f"{GATEWAY_URL}/adapters")[1])
for kind, q in (("flat", "What is this?"), ("noise", "What colour is this?")):
    b = assist(synthetic(kind), q)
    print(
        f"SYNTHETIC {kind}: status={b['status']} issues={b['issues']} score={b['unanswerable_score']} "
        f"latency_ms={b['latency_ms']} message={b['message']!r}"
    )

recs = [json.loads(line) for line in open(f"{DATA}/processed/vqa_val.jsonl")][:30]
paths = {p.name: p for p in Path(f"{LOCAL}/images").rglob("*.jpg")}
statuses, lat = {}, []
for r in recs:
    b = assist(open(paths[r["image"]], "rb").read(), r["question"])
    statuses[b["status"]] = statuses.get(b["status"], 0) + 1
    lat.append(b["latency_ms"])
print(
    "VAL SAMPLE (30, one at a time):",
    statuses,
    f"latency ms p50={statistics.median(lat):.0f} max={max(lat):.0f}",
)

**Paste back:** the `ADAPTERS`, both `SYNTHETIC` lines and the `VAL SAMPLE` line.

## 7 · Replay val and test through the served path (~35 min)
16 questions at a time, three stages each. Rows are saved to Drive as they finish; a rerun continues where it stopped.

In [ ]:
%%time
for split in ("val", "test"):
    sh(
        f"{PY} -m vizwiz.serve.replay --records {DATA}/processed --images {LOCAL}/images --split {split} --model-url {MODEL_URL} --concurrency 16 --out {OUT}/replay_{split}.jsonl"
    )

**Paste back:** the two `REPLAY DONE` lines.

## 8 · Parity and threshold tuning (~1 min)
The threshold is chosen on val only, by the rule fixed in D-060, then applied unchanged to test.

In [ ]:
sh(
    f"{PY} -m vizwiz.serve.tune --records {DATA}/processed --val-replay {OUT}/replay_val.jsonl --offline-val {OFFLINE_VAL} --test-replay {OUT}/replay_test.jsonl --report {OUT}/tune_report.json"
)
rep = json.load(open(f"{OUT}/tune_report.json"))
print("PARITY (val):", rep["parity_val"])
print("CHOSEN THRESHOLD:", rep["chosen_threshold"])
keys = (
    "vizwiz_accuracy",
    "answered_pct",
    "wrong_answers_pct_of_all",
    "wrongly_declined_pct_of_answerable",
    "missed_unanswerable_pct",
)
for split in ("val", "test"):
    for name in ("chosen", "at_0.5", "without_quality_rule", "never_decline"):
        m = rep[split][name]
        print(f"{split.upper()} {name}:", {k: m[k] for k in keys})

**Paste back:** everything this cell printed (numbers only).

## 9 · Stop the servers, flush Drive, disconnect
Upload back (summary numbers only):
- `MyDrive/vizwiz/runs/serve-check/tune_report.json`
- `MyDrive/vizwiz/runs/e5-multi-lora-val/metrics.json`

The `replay_*.jsonl` files hold model replies about dataset photos: they stay on Drive.

Then Runtime → **Disconnect and delete runtime**.

In [ ]:
stop_servers()
drive.flush_and_unmount()
print("Servers stopped, Drive flushed. Safe to disconnect.")